# 🧹 ETL — Clean → Validate → Transform

เป้าหมาย: เปลี่ยน **ข้อมูลดิบเลอะ ๆ** ให้เป็น **ข้อมูลสะอาด + ตรวจแล้ว + เสริมความหมายธุรกิจ** พร้อมเข้า star schema

> นี่คือหัวใจของงาน DE — เวลาส่วนใหญ่อยู่ตรงนี้

## โหลดข้อมูลดิบ

In [10]:
import pandas as pd
df = pd.read_csv("../data/loan_sample.csv")
print("ดิบ:", df.shape)

ดิบ: (5020, 22)


## 1️⃣ Clean — แก้ format ที่คำนวณไม่ได้

| ดิบ | หลังแก้ |
|---|---|
| `int_rate` = "13.56%" | `13.56` |
| `term` = " 36 months" | `36` |
| `emp_length` = "10+ years" | `10` |
| `issue_d` = "Dec-2018" | วันที่จริง |

In [11]:
# int_rate "13.56%" -> 13.56
df["int_rate"] = pd.to_numeric(
    df["int_rate"].astype(str).str.replace("%", "", regex=False).str.strip(),
    errors="coerce")

# term " 36 months" -> 36
df["term_months"] = pd.to_numeric(
    df["term"].astype(str).str.extract(r"(\d+)")[0], errors="coerce").astype("Int64")

# emp_length "10+ years" -> 10 , "< 1 year" -> 0
emp = df["emp_length"].astype(str).str.strip().replace(
    {"< 1 year": "0", "10+ years": "10", "n/a": None})
df["emp_length_years"] = pd.to_numeric(
    emp.str.extract(r"(\d+)")[0], errors="coerce").astype("Int64")

# issue_d "Dec-2018" -> datetime
df["issue_date"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")

df[["int_rate", "term_months", "emp_length_years", "issue_date"]].head()

,int_rate,term_months,emp_length_years,issue_date
0,15.37,60,1,2016-09-01
1,13.44,36,0,2017-03-01
2,12.06,60,2,2016-01-01
3,5.97,36,7,2016-07-01
4,18.78,36,1,2018-01-01


In [3]:
# คอลัมน์ตัวเลขอื่น ๆ -> numeric ; คอลัมน์ข้อความ -> ตัดช่องว่าง
for c in ["loan_amnt", "funded_amnt", "installment", "annual_inc", "dti",
          "fico_range_low", "fico_range_high", "total_pymnt", "total_rec_prncp", "recoveries"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

for c in ["grade", "sub_grade", "home_ownership", "verification_status",
          "purpose", "addr_state", "loan_status"]:
    df[c] = df[c].astype(str).str.strip()

df[["int_rate", "term_months", "annual_inc", "issue_date"]].dtypes

int_rate              float64
term_months             Int64
annual_inc            float64
issue_date     datetime64[us]
dtype: object

## 2️⃣ Validate + Quarantine — คัดแถวเสีย (ไม่ลบ! เก็บไว้ดูได้)

กฎ: วงเงิน > 0, รายได้ ≥ 0, มีดอกเบี้ย, งวด = 36/60, วันที่ถูก, สถานะอยู่ในลิสต์
แถวที่ไม่ผ่าน → แยกไป **quarantine** พร้อมเหตุผล

In [12]:
VALID_STATUS = {"Fully Paid", "Charged Off", "Current", "Default",
                "Late (31-120 days)", "Late (16-30 days)", "In Grace Period"}

mask = (
    (df["loan_amnt"] > 0)
    & (df["funded_amnt"] > 0)
    & (df["annual_inc"].fillna(-1) >= 0)
    & df["int_rate"].notna()
    & df["term_months"].isin([36, 60])
    & df["issue_date"].notna()
    & df["loan_status"].isin(VALID_STATUS)
)

good = df[mask].copy()
bad = df[~mask].copy()
print(f"✅ good = {len(good)}   |   ⚠️ quarantine = {len(bad)}")

✅ good = 5000   |   ⚠️ quarantine = 20


In [5]:
# ใส่ "เหตุผล" ให้แถวที่ถูกคัดออก -> audit ได้
def reason(r):
    rs = []
    if not (r["loan_amnt"] > 0): rs.append("loan_amnt<=0")
    if not (r["funded_amnt"] > 0): rs.append("funded_amnt<=0")
    if pd.isna(r["annual_inc"]) or r["annual_inc"] < 0: rs.append("annual_inc_invalid")
    if str(r["loan_status"]) not in VALID_STATUS: rs.append("status_invalid")
    return ";".join(rs)

bad["reject_reason"] = bad.apply(reason, axis=1)
bad[["id", "loan_amnt", "funded_amnt", "annual_inc", "loan_status", "reject_reason"]].head(10)

,id,loan_amnt,funded_amnt,annual_inc,loan_status,reject_reason
5000,9000000,-1000,10000,84477.0,Fully Paid,loan_amnt<=0
5001,9000001,-1000,20000,42639.0,Charged Off,loan_amnt<=0
5002,9000002,-1000,25000,75893.0,Charged Off,loan_amnt<=0
5003,9000003,-1000,12000,99507.0,Current,loan_amnt<=0
5004,9000004,-1000,8000,22669.0,Default,loan_amnt<=0
5005,9000005,35000,35000,151833.0,UNKNOWN,status_invalid
5006,9000006,15000,15000,28306.0,UNKNOWN,status_invalid
5007,9000007,35000,35000,87381.0,UNKNOWN,status_invalid
5008,9000008,5000,5000,75884.0,UNKNOWN,status_invalid
5009,9000009,25000,25000,110916.0,UNKNOWN,status_invalid


💡 **Quarantine แทนการลบ**: ข้อมูลไม่หาย ทีมเอาไปแก้แล้ว re-run ได้ — มี audit trail ครบทุกแถว

## 3️⃣ Transform — สร้างคอลัมน์ที่มีความหมายธุรกิจ
- `status_group` : Good / Bad / In Progress
- `is_default`   : 1 ถ้าเบี้ยวหนี้ (ใช้คำนวณ default rate)
- `fico_avg`     : คะแนนเครดิตเฉลี่ย
- `profit`       : กำไรคร่าว ๆ = ยอดที่ได้คืน − ยอดที่ปล่อยกู้

In [13]:
BAD_STATUS = {"Charged Off", "Default", "Late (31-120 days)", "Late (16-30 days)"}
GOOD_STATUS = {"Fully Paid", "Current", "In Grace Period"}

def status_group(s):
    if s in BAD_STATUS: return "Bad"
    if s in GOOD_STATUS: return "Good"
    return "In Progress"

good["status_group"] = good["loan_status"].map(status_group)
good["is_default"] = good["loan_status"].isin(BAD_STATUS).astype(int)
good["fico_avg"] = good[["fico_range_low", "fico_range_high"]].mean(axis=1)
good["profit"] = good["total_pymnt"] - good["funded_amnt"]

good[["loan_status", "status_group", "is_default", "fico_avg", "profit"]].head()

,loan_status,status_group,is_default,fico_avg,profit
0,Fully Paid,Good,0,707.0,8345.89
1,Fully Paid,Good,0,762.0,4185.45
2,Current,Good,0,782.0,-3412.01
3,Fully Paid,Good,0,722.0,426.27
4,In Grace Period,Good,0,692.0,-9960.95


In [7]:
print("default rate รวม: {:.1%}".format(good["is_default"].mean()))
good["status_group"].value_counts()

default rate รวม: 26.4%


status_group
Good    3679
Bad     1321
Name: count, dtype: int64

---
✅ ได้ **`good`** (สะอาด+เสริมคอลัมน์) และ **`bad`** (quarantine)

> ทั้งหมดนี้ = โมดูล `loan_etl.clean / validate / transform` (เวอร์ชัน package ที่เก็บไว้ใช้ซ้ำ)

ไปต่อ **`03_star_schema_load.ipynb`** — ปั้นเป็น star schema แล้วโหลดเข้า MSSQL